In [ ]:
# RQ6 (concurrent clients) and RQ7 (server cores) at 128 MiB x 1 KiB (decision 25), 3 load-generator runs per
# point, 30 s windows (decision 28)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

LOADRUN = repo / "results/raw/p9-20261113/loadgen"
lg = frame.load_loadgen(LOADRUN, platform)
t = plots.build(df, status, loadgen_rows=lg)
load = loadgen.yaml.safe_load(loadgen.LOAD_PATH.read_text())
print(f"{len(lg)} load-generator runs over {lg.cell_id.nunique()} cells;", lg.groupby("experiment").size().to_dict())

In [ ]:
# Gate G7 (spec 11, test_loadgen_not_bottleneck): the load generator's CPU stays under the limit at the highest C.
rows = loadgen.read_rows(LOADRUN)
problems = loadgen.g7_problems(rows, load["loadgen_cpu_limit_pct"])
print("\n".join(problems) or f"G7 PASS: load generator under {load['loadgen_cpu_limit_pct']} % CPU at every top C")
print("load-generator CPU %, max per arm:", pd.DataFrame(rows).groupby("arm").loadgen_cpu_util.max().round(0).to_dict())

In [ ]:
# RQ6: the closed-loop curve per arm (median of 3 runs per C; the sweep stops once p95 passes 10 x the C = 1 p95),
# the saturation point (highest closed-loop throughput) and, per price pair, baseline over verifiable throughput
# at saturation. H6 expects that ratio to stay near the single-query server-time ratio.
cur = t.load_curves
closed = cur[cur.experiment == "closed"].sort_values(["arm", "clients"])
print(closed[["arm", "clients", "throughput_qps", "lat_p50_s", "lat_p95_s", "server_cpu_util", "saturated_runs", "n"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
sat = plots.saturation(cur).set_index("arm")
print("\nsaturation:\n", sat[["clients", "throughput_qps", "lat_p50_s", "lat_p95_s"]].to_string(float_format=lambda v: f"{v:.4g}"))
single = t.medians[(t.medians.db_bytes == CELL[0]) & (t.medians.record_bytes == CELL[1])].set_index("arm")
for p in plots.PRICE_PAIRS:
    if p.num in sat.index and p.den in sat.index:
        print(f"{p.label}: saturation throughput ratio (baseline / verifiable) "
              f"{sat.throughput_qps[p.den] / sat.throughput_qps[p.num]:.3f}; single-query server_online_s ratio "
              f"(verifiable / baseline) {single.server_online_s[p.num] / single.server_online_s[p.den]:.3f}")

In [ ]:
# RQ6, open loop: p50 and p95 latency per offered load (fractions of each arm's saturation throughput).
op = cur[cur.experiment == "open"].sort_values(["arm", "offered_load_qps"])
print(op[["arm", "point", "offered_load_qps", "throughput_qps", "lat_p50_s", "lat_p95_s", "saturated_runs", "n"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
# RQ7: throughput against server workers W at C = 16, the speedup over W = 1, and per price pair the overhead
# (baseline over verifiable throughput) at each W. H7: within +-20 % across W when verification is client-side.
mc = cur[cur.experiment == "multicore"].sort_values(["arm", "workers"])
w1 = mc[mc.workers == 1].set_index("arm").throughput_qps
mc = mc.assign(speedup=mc.throughput_qps / mc.arm.map(w1))
print(mc[["arm", "workers", "throughput_qps", "speedup", "lat_p50_s", "server_cpu_util"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
cs = plots.core_scaling(cur)
print("\n", cs.pivot_table(index="pair", columns="workers", values="overhead").round(3).to_string())
spread = cs.groupby("pair").overhead.agg(lambda s: s.max() / s.min() - 1)
print("\nspread of the overhead across W (max / min - 1):", (100 * spread).round(1).to_dict())

In [ ]:
# F10 (latency against throughput) and F11 (throughput and overhead against W).
for name in ("F10_load_latency", "F11_cores"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F1[01]_*")))